# RNN/LSTM Sentiment Analysis

Investigating how LSTM performance on sentiment classification varies with input text length.
Trained on a combined dataset of tweets, Yelp reviews, Amazon reviews, and IMDB reviews,
bucketed by text length (super_short, short, medium, long).

In [ ]:
import os
import re
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PREFER_MPS = False
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif PREFER_MPS and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

## Data Loading

In [ ]:
DATA_DIR = "Datasets"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train_dataset.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val_dataset.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test_dataset.csv"))

BUCKETS = ["super_short", "short", "medium", "long"]
bucket_test_dfs = {
    bucket: pd.read_csv(os.path.join(DATA_DIR, f"test_{bucket}.csv"))
    for bucket in BUCKETS
}

LABEL_MAP = {"positive": 0, "neutral": 1, "negative": 2}
NUM_CLASSES = len(LABEL_MAP)

for df in [train_df, val_df, test_df, *bucket_test_dfs.values()]:
    df["label"] = df["sentiment"].map(LABEL_MAP)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(f"\nTrain label distribution:\n{train_df['sentiment'].value_counts()}")
print(f"\nTrain bucket distribution:\n{train_df['bucket'].value_counts()}")
print(f"\nBucket test set sizes:")
for b, bdf in bucket_test_dfs.items():
    print(f"  {b}: {len(bdf):,}")

## Tokenizer and Vocabulary

In [ ]:
PAD_IDX = 0
UNK_IDX = 1
MIN_FREQ = 2


def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()


def build_vocab(texts: pd.Series, min_freq: int = MIN_FREQ) -> dict[str, int]:
    counts = Counter()
    for text in texts:
        counts.update(tokenize(str(text)))

    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    idx = 2
    for word, freq in counts.most_common():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab


def encode_text(text: str, vocab: dict[str, int]) -> list[int]:
    return [vocab.get(tok, UNK_IDX) for tok in tokenize(str(text))]


vocab = build_vocab(train_df["text"])
print(f"Vocabulary size: {len(vocab):,}")

## Load GloVe Embeddings

In [ ]:
import urllib.request
import zipfile

GLOVE_DIM = 100
GLOVE_DIR = "glove"
GLOVE_FILE = os.path.join(GLOVE_DIR, f"glove.6B.{GLOVE_DIM}d.txt")
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"

if not os.path.exists(GLOVE_FILE):
    os.makedirs(GLOVE_DIR, exist_ok=True)
    zip_path = os.path.join(GLOVE_DIR, "glove.6B.zip")

    if not os.path.exists(zip_path):
        print("Downloading GloVe (~862 MB, this takes a couple minutes)...")
        urllib.request.urlretrieve(GLOVE_URL, zip_path)
        print("Download complete.")

    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extract(f"glove.6B.{GLOVE_DIM}d.txt", GLOVE_DIR)
    print("Done.")
else:
    print(f"GloVe already available at {GLOVE_FILE}")


def load_glove(path: str) -> dict[str, np.ndarray]:
    embeddings = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.array(parts[1:], dtype=np.float32)
            embeddings[word] = vec
    return embeddings


def build_embedding_matrix(vocab: dict[str, int], glove: dict[str, np.ndarray], dim: int):
    matrix = np.zeros((len(vocab), dim), dtype=np.float32)
    found, missing = 0, 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
            found += 1
        else:
            
            matrix[idx] = np.random.uniform(-0.25, 0.25, dim)
            missing += 1

    matrix[PAD_IDX] = 0.0
    print(f"Embedding matrix: {found:,} found in GloVe, {missing:,} missing (randomly initialized)")
    return torch.tensor(matrix)


glove = load_glove(GLOVE_FILE)
pretrained_embeddings = build_embedding_matrix(vocab, glove, GLOVE_DIM)
del glove 

## PyTorch Datasets and DataLoaders

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts: pd.Series, labels: pd.Series, vocab: dict[str, int]):
        self.encoded = [encode_text(t, vocab) for t in texts]
        self.labels = labels.values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.encoded[idx], dtype=torch.long), self.labels[idx]


def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)

    lengths = lengths.clamp(min=1)

    padded = torch.zeros(len(seqs), lengths.max().item(), dtype=torch.long)
    for i, seq in enumerate(seqs):
        if len(seq) > 0:
            padded[i, : len(seq)] = seq

    labels = torch.tensor(labels, dtype=torch.long)

    sorted_idx = lengths.argsort(descending=True)
    return padded[sorted_idx], lengths[sorted_idx], labels[sorted_idx]


BATCH_SIZE = 64


def make_loader(df: pd.DataFrame, vocab: dict, shuffle: bool = False) -> DataLoader:
    ds = SentimentDataset(df["text"], df["label"], vocab)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate_fn)


train_loader = make_loader(train_df, vocab, shuffle=True)
val_loader = make_loader(val_df, vocab)
test_loader = make_loader(test_df, vocab)

bucket_test_loaders = {
    bucket: make_loader(bdf, vocab)
    for bucket, bdf in bucket_test_dfs.items()
}

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")

## LSTM Model

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 100,
        hidden_dim: int = 256,
        num_classes: int = 3,
        num_layers: int = 2,
        dropout: float = 0.5,
        bidirectional: bool = True,
        pretrained_embeddings: torch.Tensor = None,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.dropout = nn.Dropout(dropout)
        num_directions = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * num_directions, num_classes)

    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=True
        )
        _, (hidden, _) = self.lstm(packed)

        if self.lstm.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]

        return self.fc(self.dropout(hidden))

## Training and Evaluation

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, log_every=100):
    model.train()
    total_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (padded, lengths, labels) in enumerate(loader, start=1):
        padded, labels = padded.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(padded, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)

        if batch_idx % log_every == 0 or batch_idx == len(loader):
            elapsed = time.time() - epoch_start
            avg_loss = total_loss / min(batch_idx * loader.batch_size, len(loader.dataset))
            print(
                f"    batch {batch_idx:>3}/{len(loader)} | avg_loss={avg_loss:.4f} | elapsed={elapsed:.1f}s",
                flush=True,
            )

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for padded, lengths, labels in loader:
        padded = padded.to(device)
        logits = model(padded, lengths)
        preds = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro"),
        "macro_recall": recall_score(y_true, y_pred, average="macro"),
    }


class EarlyStopping:
    def __init__(self, patience: int = 3):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model) -> bool:
        """Returns True if training should stop."""
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

In [ ]:
EMBED_DIM = GLOVE_DIM
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.5
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 15
PATIENCE = 3
LOG_EVERY = 100


def build_model():
    return LSTMClassifier(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        bidirectional=True,
        pretrained_embeddings=pretrained_embeddings,
    ).to(DEVICE)


def train_model(model, train_loader, val_loader, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Full training loop with early stopping and LR scheduling."""
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=1,
    )
    criterion = nn.CrossEntropyLoss()
    stopper = EarlyStopping(patience=patience)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, max_epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch {epoch}/{max_epochs}  (lr={current_lr:.1e})")
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            DEVICE,
            log_every=LOG_EVERY,
        )

        model.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for padded, lengths, labels in val_loader:
                padded, labels = padded.to(DEVICE), labels.to(DEVICE)
                logits = model(padded, lengths)
                val_loss_total += criterion(logits, labels).item() * len(labels)
        val_loss = val_loss_total / len(val_loader.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        scheduler.step(val_loss)

        print(f"  epoch_summary | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}", flush=True)

        if stopper.step(val_loss, model):
            print(f"  Early stopping at epoch {epoch}", flush=True)
            break

    stopper.restore(model)
    return history

## Experiment 1: Train on Full Dataset, Evaluate Per Bucket

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: Full-dataset LSTM")
print("=" * 60)

full_model = build_model()
full_history = train_model(full_model, train_loader, val_loader)

In [ ]:
label_names = ["positive", "neutral", "negative"]

y_pred, y_true = predict(full_model, test_loader, DEVICE)
full_overall_metrics = compute_metrics(y_true, y_pred)
print("Overall test set:")
print(classification_report(y_true, y_pred, target_names=label_names))

full_bucket_metrics = {}
for bucket in BUCKETS:
    y_pred, y_true = predict(full_model, bucket_test_loaders[bucket], DEVICE)
    metrics = compute_metrics(y_true, y_pred)
    full_bucket_metrics[bucket] = metrics
    print(f"\n--- {bucket} ---")
    print(classification_report(y_true, y_pred, target_names=label_names))

## Experiment 2: Train Per Bucket, Evaluate Per Bucket

In [ ]:
per_bucket_metrics = {}

for bucket in BUCKETS:
    print("=" * 60)
    print(f"EXPERIMENT 2: Training on '{bucket}' bucket only")
    print("=" * 60)

    bucket_train_df = train_df[train_df["bucket"] == bucket].reset_index(drop=True)
    bucket_val_df = val_df[val_df["bucket"] == bucket].reset_index(drop=True)

    bucket_train_loader = make_loader(bucket_train_df, vocab, shuffle=True)
    bucket_val_loader = make_loader(bucket_val_df, vocab)

    model = build_model()
    train_model(model, bucket_train_loader, bucket_val_loader)

    y_pred, y_true = predict(model, bucket_test_loaders[bucket], DEVICE)
    metrics = compute_metrics(y_true, y_pred)
    per_bucket_metrics[bucket] = metrics

    print(f"\nTest results for '{bucket}':")
    print(classification_report(y_true, y_pred, target_names=label_names))
    print()

## Results

In [ ]:
os.makedirs("results", exist_ok=True)

rows = []
for bucket in BUCKETS:
    fm = full_bucket_metrics[bucket]
    pm = per_bucket_metrics[bucket]
    rows.append({
        "bucket": bucket,
        "full_accuracy": fm["accuracy"],
        "full_macro_f1": fm["macro_f1"],
        "perbucket_accuracy": pm["accuracy"],
        "perbucket_macro_f1": pm["macro_f1"],
    })

results_df = pd.DataFrame(rows)
results_df = results_df.set_index("bucket")
print(results_df.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(BUCKETS))
width = 0.35

bars1a = axes[0].bar(x - width / 2, results_df["full_accuracy"], width, label="Full-data model")
bars1b = axes[0].bar(x + width / 2, results_df["perbucket_accuracy"], width, label="Per-bucket model")
for bar in list(bars1a) + list(bars1b):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy by Text Length Bucket")
axes[0].set_xticks(x)
axes[0].set_xticklabels(BUCKETS, rotation=15)
axes[0].legend()
axes[0].set_ylim(0, 1.05)

bars2a = axes[1].bar(x - width / 2, results_df["full_macro_f1"], width, label="Full-data model")
bars2b = axes[1].bar(x + width / 2, results_df["perbucket_macro_f1"], width, label="Per-bucket model")
for bar in list(bars2a) + list(bars2b):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Macro F1 by Text Length Bucket")
axes[1].set_xticks(x)
axes[1].set_xticklabels(BUCKETS, rotation=15)
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("results/lstm_accuracy_f1_by_bucket.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, bucket in zip(axes, BUCKETS):
    y_pred, y_true = predict(full_model, bucket_test_loaders[bucket], DEVICE)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(f"{bucket}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

fig.suptitle("Confusion Matrices — Full-Data LSTM by Bucket", y=1.02)
plt.tight_layout()
plt.savefig("results/lstm_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
epochs = range(1, len(full_history["train_loss"]) + 1)
ax.plot(epochs, full_history["train_loss"], marker="o", label="Train Loss")
ax.plot(epochs, full_history["val_loss"], marker="s", label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Full-Data LSTM Training Curve")
ax.legend()
plt.tight_layout()
plt.savefig("results/lstm_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Export Results

In [ ]:
export_rows = []
for bucket in BUCKETS:
    fm = full_bucket_metrics[bucket]
    pm = per_bucket_metrics[bucket]
    export_rows.append({
        "model": "LSTM",
        "experiment": "full_dataset",
        "bucket": bucket,
        **fm,
    })
    export_rows.append({
        "model": "LSTM",
        "experiment": "per_bucket",
        "bucket": bucket,
        **pm,
    })

export_rows.append({
    "model": "LSTM",
    "experiment": "full_dataset",
    "bucket": "all",
    **full_overall_metrics,
})

export_df = pd.DataFrame(export_rows)
export_df.to_csv("results/rnn_lstm_results.csv", index=False)
print("Saved results/rnn_lstm_results.csv")
print(export_df.round(4).to_string(index=False))